# This notebook is for experimenting specifically on Target_LTV continuous Regression. It has no Classification part. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.paths import FEATURES_TARGETS
from src.models.ltv_model import create_ltv_pipeline
from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

In [2]:
df = pd.read_csv(FEATURES_TARGETS)
y_ltv = df["Target_LTV"]
X = df.drop(columns=["Target_LTV", "Target_churn", "Customer ID"])

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y_ltv, test_size=0.2, random_state=42)

# But we only need those y_ltv where y_ltv > 0 because y_ltv == 0 means customer has already churned. Hence we need to estimate LTV for customer who still exist.

X_train = X_train[y_train>0]
y_train = y_train[y_train>0]

X_test = X_test[y_test>0]
y_test = y_test[y_test>0]


In [4]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(2186, 5)
(2186,)
(540, 5)
(540,)


<h3>Starting with Models part now </h3>
    For metrics will be using 3 metrics to evaluate the models

    1) MAE - This will be our primary Metric.
    2) RMSE - This punishes large errors heavily, so it will be helpful to determine sensitivity to big errors.
    3) R2 - It will tell how much variation in LTV is explained by model, not a very important one for this.

In [5]:
cv_scores_avg_dict = {}

def cv_scores_avg(cv_results_dict: dict, model_name: str, push_to_dict=True):

    test_mae = -cv_results_dict["test_mae"]
    test_rmse = -cv_results_dict["test_rmse"]
    test_r2 = cv_results_dict["test_r2"]

    avg_test_mae = test_mae.mean()
    avg_test_rmse = test_rmse.mean()
    avg_test_r2 = test_r2.mean()

    scores_dict = {"Avg_MAE":avg_test_mae, "Avg_RMSE": avg_test_rmse, "Avg_R2": avg_test_r2}

    if push_to_dict:
        cv_scores_avg_dict[model_name] = scores_dict

    return scores_dict

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}

<h3>Starting the Modelling part</h3>

<h3>Linear Models</h3>

<h5>Linear Regression (Baseline Model)</h5>

In [6]:
linear_model = create_ltv_pipeline(LinearRegression())
cv_results_linear_regression = cross_validate(estimator=linear_model, X=X_train, y=y_train, cv=5, scoring=scoring)
cv_results_linear_regression

print(cv_scores_avg(cv_results_linear_regression, "LinearRegression", True))


{'Avg_MAE': np.float64(1601.4722088910362), 'Avg_RMSE': np.float64(7275.868855416469), 'Avg_R2': np.float64(0.456279708380921)}


1) Avg_MAE of 1601 suggests that model is off from original values by +-1601. 
2) Avg_RMSE of 7275 suggests that model is making predictions which are way off for some customers. 
3) Avg_R2 of 0.456 tells that LinearRegression explains 45.6% variation in Data

Reason for high RMSE :-

In notebook 01_eda_and_diagnostics we discovered that Target_LTV has mean=2728, median=893, 75th quantile=2262 but it has maximum of 287491 which is creating issue of high RMSE in this model. 

<h5>Ridge Regression</h5>

In [7]:
ridge_model = create_ltv_pipeline(Ridge())
cv_results_ridge = cross_validate(ridge_model, X=X_train, y=y_train, cv=5, scoring=scoring)

print(cv_scores_avg(cv_results_ridge, "RidgeRegression", True))
cv_scores_avg_dict

{'Avg_MAE': np.float64(1602.0759691010946), 'Avg_RMSE': np.float64(7282.8653267808895), 'Avg_R2': np.float64(0.4554438674785907)}


{'LinearRegression': {'Avg_MAE': np.float64(1601.4722088910362),
  'Avg_RMSE': np.float64(7275.868855416469),
  'Avg_R2': np.float64(0.456279708380921)},
 'RidgeRegression': {'Avg_MAE': np.float64(1602.0759691010946),
  'Avg_RMSE': np.float64(7282.8653267808895),
  'Avg_R2': np.float64(0.4554438674785907)}}

<h3>Tree Models</h3>

<h5>Decision Tree</h5>

In [8]:
decision_tree_model = create_ltv_pipeline(DecisionTreeRegressor(random_state=42))

cv_results_decision_tree = cross_validate(estimator=decision_tree_model, X=X_train, y=y_train, cv=5, scoring=scoring)

print(cv_scores_avg(cv_results_decision_tree, "DecisionTree", True))

{'Avg_MAE': np.float64(2197.8418405483635), 'Avg_RMSE': np.float64(8681.526690020482), 'Avg_R2': np.float64(0.21613980260362683)}


<h3>Ensemble Models</h3>

<h5>Random Forest</h5>

In [9]:
random_forest_model = create_ltv_pipeline(RandomForestRegressor(random_state=42))
cv_results_random_forest = cross_validate(estimator=random_forest_model, X=X_train, y=y_train, cv=5, scoring=scoring)
print(cv_scores_avg(cv_results_random_forest, "RandomForest", True))

{'Avg_MAE': np.float64(1648.773367481085), 'Avg_RMSE': np.float64(7433.201895296891), 'Avg_R2': np.float64(0.45532967749759184)}


<h5>Gradient Boosting</h5>

In [10]:
gradient_boosting_model = create_ltv_pipeline(GradientBoostingRegressor(random_state=42))

cv_results_gradient_boosting = cross_validate(estimator=gradient_boosting_model, X=X_train, y=y_train, cv=5, scoring=scoring)
print(cv_scores_avg(cv_results_gradient_boosting, "GradientBoosting", True))

{'Avg_MAE': np.float64(1601.360131947498), 'Avg_RMSE': np.float64(7168.317267825206), 'Avg_R2': np.float64(0.5065056667875493)}


<h5>Hist Gradient Booster</h5>

In [11]:
hist_gradient_model = create_ltv_pipeline(HistGradientBoostingRegressor(random_state=42))

cv_results_hist_gradient = cross_validate(estimator=hist_gradient_model, X=X_train, y=y_train, cv=5, scoring=scoring)

print(cv_scores_avg(cv_results_hist_gradient, "HistGradientBoosting", True))


{'Avg_MAE': np.float64(1664.9732690550456), 'Avg_RMSE': np.float64(7708.233970130759), 'Avg_R2': np.float64(0.4363676002902733)}


In [12]:
cv_scores_df = pd.DataFrame(cv_scores_avg_dict).T
cv_scores_df.sort_values(by=["Avg_MAE"], ascending=True)

,Avg_MAE,Avg_RMSE,Avg_R2
GradientBoosting,1601.360132,7168.317268,0.506506
LinearRegression,1601.472209,7275.868855,0.456280
RidgeRegression,1602.075969,7282.865327,0.455444
RandomForest,1648.773367,7433.201895,0.455330
HistGradientBoosting,1664.973269,7708.233970,0.436368
DecisionTree,2197.841841,8681.526690,0.216140


<h3>Hyperparameter Tuning</h3>

<h4>Gradient Boosting</h4>

<h5>Experiment - 1</h5>

In [13]:
param_grid_1 = {
    "regressor__regressor__n_estimators": [50, 100, 200, 300],
    "regressor__regressor__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "regressor__regressor__max_depth": [2, 3, 4, 5],
    "regressor__regressor__min_samples_split": [2, 5, 10],
    "regressor__regressor__min_samples_leaf": [1, 2, 4],
}

#Reason for using regressor__regressor__paramname is because I am not using GradeintBoosting directly i am using it trhough function create_ltv_pipeline.
# Function create_ltv_pipeline -> prepreocessor -> Regressor 
# -> TransformedTargetRegressor -> GradientBoostingRegressor 

tuned_model_1 = create_ltv_pipeline(GradientBoostingRegressor(random_state=42))

random_search_gb_1 = RandomizedSearchCV(estimator=tuned_model_1,
                                      param_distributions=param_grid_1,
                                      n_iter=30,
                                      cv=5,
                                      scoring="neg_root_mean_squared_error",
                                      random_state=42,
                                      n_jobs=-1)

random_search_gb_1.fit(X_train, y_train)

print(random_search_gb_1.best_params_)
print(random_search_gb_1.best_score_)


{'regressor__regressor__n_estimators': 200, 'regressor__regressor__min_samples_split': 2, 'regressor__regressor__min_samples_leaf': 1, 'regressor__regressor__max_depth': 3, 'regressor__regressor__learning_rate': 0.2}
-6912.3587620806775


In [14]:
tuned_gb_1 = random_search_gb_1.best_estimator_

cv_results_best_gb_1 = cross_validate(estimator=tuned_gb_1, X=X_train, y=y_train, cv=5, scoring=scoring)
print(cv_scores_avg(cv_results_best_gb_1, "tuned_gb_1", False))

{'Avg_MAE': np.float64(1616.1673830007207), 'Avg_RMSE': np.float64(6912.3587620806775), 'Avg_R2': np.float64(0.5293598412700504)}


<h5>Experiment-2</h5>

In [15]:
'''

param_grid_2 = {
    "regressor__regressor__n_estimators": [50, 100, 200, 300, 500, 800],
    "regressor__regressor__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.15],
    "regressor__regressor__max_depth": [1, 2, 3, 4, 5],
    "regressor__regressor__min_samples_split": [2, 5, 10, 20, 30],
    "regressor__regressor__min_samples_leaf": [1, 2, 4, 8, 12, 20],
    "regressor__regressor__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "regressor__regressor__loss": ["squared_error","absolute_error","huber"]
}

tuned_model_2 = create_ltv_pipeline(GradientBoostingRegressor(random_state=42))

random_search_gb_2 = RandomizedSearchCV(estimator=tuned_model_2,
                                        param_distributions=param_grid_2,
                                        n_iter=100,
                                        cv=5,
                                        scoring="neg_root_mean_squared_error",
                                        random_state=42,
                                        n_jobs=-1)

random_search_gb_2.fit(X_train, y_train)
print(random_search_gb_2.best_params_)
print(random_search_gb_2.best_score_)

'''

'\n\nparam_grid_2 = {\n    "regressor__regressor__n_estimators": [50, 100, 200, 300, 500, 800],\n    "regressor__regressor__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.15],\n    "regressor__regressor__max_depth": [1, 2, 3, 4, 5],\n    "regressor__regressor__min_samples_split": [2, 5, 10, 20, 30],\n    "regressor__regressor__min_samples_leaf": [1, 2, 4, 8, 12, 20],\n    "regressor__regressor__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],\n    "regressor__regressor__loss": ["squared_error","absolute_error","huber"]\n}\n\ntuned_model_2 = create_ltv_pipeline(GradientBoostingRegressor(random_state=42))\n\nrandom_search_gb_2 = RandomizedSearchCV(estimator=tuned_model_2,\n                                        param_distributions=param_grid_2,\n                                        n_iter=100,\n                                        cv=5,\n                                        scoring="neg_root_mean_squared_error",\n                                        random_state=42,\n          

In [16]:
# tuned_gb_2 = random_search_gb_2.best_estimator_

# cv_results_best_gb_2 = cross_validate(estimator=tuned_gb_2, X=X_train, y=y_train, cv=5, scoring=scoring)
# print(cv_scores_avg(cv_results_best_gb_2, "tuned_gb_2", False))


<b>Choosing Experiment - 1 Gradient Boosting Algorithm to move foreward.</b>

<small>Why Experiment-1 Gradient Boosting Model ?

Baseline GB Model has 
1) Avg_MAE - 1601
2) Avg_RMSE - 7168
3) Avg_R2 - 0.506


Experiment-1 GB Model has 
1) Avg_MAE - 1616
2) Avg_RMSE - 6912
3) Avg_R2 - 0.529

Though Experiment-1 GB model increase MAE by approximately by 15 but it reduced RMSE substantially by approx 256. This indicates that experiment-1 GB Model handles large predcition errors much better, while maintaing similar avg_mae</small>


<h3>Final Tests</h3>

<h5>Training on Full Training Datset and then testing on Test set</h5>

In [32]:
best_model = tuned_gb_1 # Experiment-1 GB Model

tuned_gb_1.fit(X_train, y_train)
y_pred_best_model = tuned_gb_1.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_best_model)
rmse = root_mean_squared_error(y_test, y_pred_best_model)
r2 = r2_score(y_test, y_pred_best_model)

print(f"MAE: {mae}\nRMSE: {rmse}\nR2:{r2}")

MAE: 1557.9164444060784
RMSE: 8635.579136397675
R2:0.5468289965709219


The model predicts ordinary customers very well, avg_MAE of 1557 is evidence for that. But for Extreme LTV values it is struggling RMSE of 8635

In [38]:
# Because of high RMSE testing for high_values in Target_LTV which are more than 90th quantile

best_model_outputs_df = pd.DataFrame({
    "Actual_LTV": y_test,
    "Predicted_LTV": y_pred_best_model
})

threshold = y_train.quantile(0.90)
best_model_high_values = best_model_outputs_df[best_model_outputs_df["Actual_LTV"] >= threshold]

best_model_high_values

,Actual_LTV,Predicted_LTV
2987,6748.80,1339.365166
1954,5220.75,5987.009475
2340,12849.38,7164.130529
4026,59824.48,60118.310785
2096,4997.10,5533.260404
...,...,...
429,5655.91,5893.046002
665,11604.83,4938.256487
1055,6403.25,574.590349
2240,14293.12,9151.406916


In [39]:
best_model_high_values_mae = mean_absolute_error(
    best_model_high_values["Actual_LTV"],
    best_model_high_values["Predicted_LTV"]
)

best_model_high_values_rmse = root_mean_squared_error(
    best_model_high_values["Actual_LTV"],
    best_model_high_values["Predicted_LTV"]
)

print("High-value MAE:", best_model_high_values_mae)
print("High-value RMSE:", best_model_high_values_rmse)

High-value MAE: 7688.247076096268
High-value RMSE: 24745.346895650215


<h3> New Experiments </h3> 

We have np.log1p() transformation applied to target_LTV inside create_ltv_pipeline going to turn that off for few experiments and train on raw target_LTV without any transformation

In [20]:
raw_gb_model = create_ltv_pipeline(model=GradientBoostingRegressor(random_state=42), transform_target=False)

cv_results_raw_gb_model = cross_validate(estimator=raw_gb_model, X=X_train, y=y_train, cv=5, scoring=scoring)

print(cv_scores_avg(cv_results_raw_gb_model, "GBModelNoTargetTransformation", False))

{'Avg_MAE': np.float64(1724.2781134442764), 'Avg_RMSE': np.float64(7230.386702942871), 'Avg_R2': np.float64(0.4914531386353895)}


<small>

Without np.log1p() transformation of Target_LTV we have :-

1) Avg_MAE - 1724
2) Avg_RMSE - 7230
3) Avg_R2 - 0.491

With np.log1p() transformed Target_LTV the metrics for tuned_gb_1 model were :-

1) Avg_MAE - 1616
2) Avg_RMSE - 6912
3) Avg_R2 - 0.239

These metrics are lower than metrics of tuned_gb_1 model. So we are sure that np.log1p() transformation is helping. Hence dropping this model 
</small>

<h5> Trying Sample Weighting </h5>

In [ ]:
high_value_threshold = y_train.quantile(0.90)
sample_weights = np.where(y_train >= high_value_threshold, 3.0, 1.0)
y_train = y_train*sample_weights

3963      311.26
192      4111.26
2159      596.85
1842    24411.06
4010      207.50
          ...   
3171     1721.78
466     18494.61
3092      183.70
3772      993.18
860       678.01
Name: Target_LTV, Length: 2186, dtype: float64

In [59]:
weighted_gb_model_1 = create_ltv_pipeline(GradientBoostingRegressor(random_state=42), True)

# weighted_gb_model_1.fit(X_train,y_train,regressor__sample_weight=sample_weights)

# y_pred_w_gb_model_1 = weighted_gb_model_1.predict(X_test)

# mae = mean_absolute_error(y_test, y_pred_w_gb_model_1)
# rmse = root_mean_squared_error(y_test, y_pred_w_gb_model_1)
# r2 = r2_score(y_test, y_pred_w_gb_model_1)

# print("MAE:", mae)
# print("RMSE:", rmse)
# print("R2:", r2)


cv_results_w_gb_model_1 = cross_validate(estimator=weighted_gb_model_1, X=X_train, y=y_train, cv=5, scoring=scoring)
print(cv_scores_avg(cv_results_w_gb_model_1, "WeightedGBModel1", False))


{'Avg_MAE': np.float64(4269.505972299065), 'Avg_RMSE': np.float64(23795.831718705947), 'Avg_R2': np.float64(0.4035550845308659)}


Comparing weighted GB Model 1 with Experiment-1 GB Model which is best_model = tuned_gb_1

<small>

For Experiment-1 GB Model 

1) MAE: 1557.9164444060784
2) RMSE: 8635.579136397675
3) R2: 0.5468289965709219

For weighted_gb_model_1

1) MAE: 1442.19215967014
2) RMSE: 7855.992868648763
3) R2: 0.6249567636810207

</small>

In [ ]:
w_gb_model_1_outputs_df = pd.DataFrame({
    "Actual_LTV": y_test,
    "Predicted_LTV": y_pred_w_gb_model_1
})

threshold = y_train.quantile(0.90)
w_gb_model_1_high_values = w_gb_model_1_outputs_df[w_gb_model_1_outputs_df["Actual_LTV"] >= threshold]

w_gb_model_1_high_values

,Actual_LTV,Predicted_LTV
2987,6748.80,1271.032111
1954,5220.75,7026.575566
2340,12849.38,7752.964376
4026,59824.48,56771.895104
2096,4997.10,7146.180121
...,...,...
429,5655.91,5880.348651
665,11604.83,6902.263560
1055,6403.25,898.086993
2240,14293.12,9635.582714


In [44]:
w_gb_model_1_high_mae = mean_absolute_error(
    w_gb_model_1_high_values["Actual_LTV"],
    w_gb_model_1_high_values["Predicted_LTV"]
)

w_gb_model_1_high_rmse = root_mean_squared_error(
    w_gb_model_1_high_values["Actual_LTV"],
    w_gb_model_1_high_values["Predicted_LTV"]
)

print("High-value MAE:", w_gb_model_1_high_mae)
print("High-value RMSE:", w_gb_model_1_high_rmse)

High-value MAE: 6358.11414136249
High-value RMSE: 22545.954781542318


In [51]:
# Trying more experiments with sample weights 

sample_weights = np.ones(len(y_train))

sample_weights[y_train >= y_train.quantile(0.95)] = 5
sample_weights[y_train >= y_train.quantile(0.90)] = 2

In [52]:
weighted_gb_model_2 = create_ltv_pipeline(GradientBoostingRegressor(random_state=42), True)

weighted_gb_model_2.fit(X_train,y_train,regressor__sample_weight=sample_weights)

y_pred_w_gb_model_2 = weighted_gb_model_2.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_w_gb_model_2)
rmse = root_mean_squared_error(y_test, y_pred_w_gb_model_2)
r2 = r2_score(y_test, y_pred_w_gb_model_2)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 1476.4430799315746
RMSE: 7552.116986071966
R2: 0.6534095482706629


In [53]:
w_gb_model_2_outputs_df = pd.DataFrame({
    "Actual_LTV": y_test,
    "Predicted_LTV": y_pred_w_gb_model_2
})

threshold = y_train.quantile(0.90)
w_gb_model_2_high_values = w_gb_model_2_outputs_df[w_gb_model_2_outputs_df["Actual_LTV"] >= threshold]

w_gb_model_2_high_values

,Actual_LTV,Predicted_LTV
2987,6748.80,1287.572897
1954,5220.75,6777.206093
2340,12849.38,6777.206093
4026,59824.48,51424.295594
2096,4997.10,6675.030816
...,...,...
429,5655.91,5634.068232
665,11604.83,6652.144925
1055,6403.25,903.962493
2240,14293.12,9297.675064


In [54]:
w_gb_model_2_high_mae = mean_absolute_error(
    w_gb_model_2_high_values["Actual_LTV"],
    w_gb_model_2_high_values["Predicted_LTV"]
)

w_gb_model_2_high_rmse = root_mean_squared_error(
    w_gb_model_2_high_values["Actual_LTV"],
    w_gb_model_2_high_values["Predicted_LTV"]
)

print("High-value MAE:", w_gb_model_2_high_mae)
print("High-value RMSE:", w_gb_model_2_high_rmse)


High-value MAE: 6918.592422775226
High-value RMSE: 21695.915270299087


<small>

For tuned_gb_1 model where we did not apply any weights to y_train (Target_LTV) values the metrics for high_values which are in 90th quantile are 

1) High-value MAE: 7688.247076096268
2) High-value RMSE: 24745.346895650215

For weighted_gb_model_1 where we gave weights to y_train which are if y_train >= 90th quantile then its 3 else its 1. The metrics for experiment are 

1) High-value MAE: 6358.11414136249
2) High-value RMSE: 22545.954781542318

For weighted_gb_model_2 where we gave different weights for values in 95th quantile got 5, values in 90th quantile got 2 and rest all values were given 1. The metrics for high_values (in 90th percentile are)

1) High-value MAE: 6918.592422775226
2) High-value RMSE: 21695.915270299087

<b>Sample weighting improved high-value customer predictions substantially. Compared with the unweighted model, both weighted experiments reduced high-value MAE and RMSE. Experiment 2 achieved the lowest high-value RMSE, indicating that it was better at reducing large prediction errors among high-value customers, while Experiment 1 achieved the lowest high-value MAE.</b>

</small>